# 06 — Trivial baselines

What is the floor? If a frozen ImageNet backbone with a linear head reaches within a point or two
of the deep models under honest evaluation, then "deep learning solves this dataset" is not a
supportable claim, and that reframes the whole benchmark.

Budget < 1 GPU-h.

In [ ]:
# ── environment ──────────────────────────────────────────────────────────
# 1. Add the dataset:  hamdallak/the-iqothnccd-lung-cancer-dataset
# 2. Add your uploaded library dataset: iqoth-honest-lib  (upload src/iqoth/)
import sys, os, json, time, types

print("Mounted in /kaggle/input:", os.listdir("/kaggle/input") if os.path.isdir("/kaggle/input") else "no /kaggle/input")

LIB = None
ROOT = None
if os.path.isdir("/kaggle/input"):
    for root, dirs, files in os.walk("/kaggle/input"):
        if "splits.py" in files and "audit.py" in files and not LIB:
            LIB = root
        if any("bengin" in d.lower() or "benign" in d.lower() for d in dirs) and not ROOT:
            ROOT = root

if LIB:
    pkg = types.ModuleType("iqoth")
    pkg.__path__ = [LIB]
    sys.modules["iqoth"] = pkg
    sys.path.insert(0, LIB)
else:
    sys.path.insert(0, "src")          # local fallback

if not ROOT:
    ROOT = os.environ.get("IQOTH_ROOT")

assert os.path.isdir(ROOT or ""), f"dataset not found; /kaggle/input tree: {list(os.walk('/kaggle/input')) if os.path.isdir('/kaggle/input') else 'no input'}"

RESULTS = "/kaggle/working/results"
os.makedirs(RESULTS, exist_ok=True)

from iqoth import splits, audit, metrics
print("lib:", LIB)
print("dataset:", ROOT)

import torch
from iqoth import engine, models, data
print("torch", torch.__version__, "| cuda", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

In [ ]:
import numpy as np
OUT = f"{RESULTS}/06_baselines"; os.makedirs(OUT, exist_ok=True)

# Majority-class baseline, computed directly from the split -- no training.
maj_results = {}
for strat in ["random_image", "block_shuffled"]:
    recs = splits.make_split(splits.build_registry(ROOT), strat, seed=42)
    tr = [r["label"] for r in recs if r["split"] == "train"]
    te = [r["label"] for r in recs if r["split"] == "test"]
    maj = max(set(tr), key=tr.count)
    yp = np.full(len(te), maj)
    s = metrics.summarise_run(np.array(te), yp)
    maj_results[strat] = s
    print(f"majority({strat:<16}) acc {s['accuracy']:6.2f}  macroF1 {s['macro_f1']:6.2f}  "
          f"BenignF1 {s['benign_f1']:6.2f}")
with open(f"{OUT}/majority_baselines.json", "w") as f:
    json.dump(maj_results, f, indent=2)


In [ ]:
# Linear probe on frozen ResNet-50 across 20 seeds x 2 strategies (40 runs)
SEEDS = [42, 123, 2024, 7, 2718, 1, 2, 3, 4, 5, 6, 8, 9, 10, 11, 12, 13, 14, 15, 16]
for strat in ["random_image", "block_shuffled"]:
    for seed in SEEDS:
        if os.path.exists(f"{OUT}/linear_probe__{strat}__seed{seed}_result.json"):
            continue
        engine.run_experiment(ROOT, "linear_probe", strat, seed, OUT)


In [ ]:
import statistics as st
by = {}
for r in engine.collect_results(OUT):
    if r.get("model") == "linear_probe":
        by.setdefault(r["strategy"], []).append(r["accuracy"])
print("")
print("=== Linear Probe (Frozen ResNet-50) 20-Seed Results ===")
for s, v in by.items():
    print(f"linear_probe {s:<18} n={len(v):2d} | Mean Acc: {st.mean(v):6.2f}% +- {st.stdev(v) if len(v)>1 else 0:.2f}% | Median: {st.median(v):6.2f}%")
